# Piloto-v3 — Ordem: C2 → guard check → 28 experimentos

**Configuracao obrigatoria antes de salvar e rodar:**
1. Settings -> Accelerator -> **GPU T4 x2**
2. Settings -> Internet -> **On**
3. Settings -> Secrets -> adicione `GITHUB_TOKEN` e `HF_TOKEN` (**marque "Attach to notebook"** para cada um)
4. Para retomar: Add Data -> Your Work -> versao anterior -> **Add as input**
5. Para rodar: Save Version -> **Save & Run All** (nao Run All Cells)

**Ordem de execucao:**
- Cel 8: Carrega manifesto
- Cel 9: **(a)** extracao livre treino → **(b)** definicoes D-24 → **(c)** inducao com EDC → **(d)** tabela calibracao → push C2
- Cel 10: **Guard check** — para com erro claro se C2 estiver ausente ou for de outra sessao
- Cel 11: 28 experimentos (C2 ja pronto)
- Cel 12: Push final

**Tempo estimado:** ~55 min (inducao C2 ~12 min + 28 exp ~33 min + definicoes ~10 min)

In [ ]:
# Celula 1 - Configuracao
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret('GITHUB_TOKEN')
HF_TOKEN     = secrets.get_secret('HF_TOKEN')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

REPO_SLUG    = 'felipemoraissoares/does-ontology-matter'
REPO_URL     = f'https://{GITHUB_TOKEN}@github.com/{REPO_SLUG}.git'
BENCH_URL    = 'https://github.com/cenguix/Text2KGBench.git'
BENCH_COMMIT = '50a3d255371b8817cdff70fd88459ac82b339cfe'

WORK_DIR   = '/kaggle/working'
REPO_DIR   = f'{WORK_DIR}/does-ontology-matter'
BENCH_DIR  = f'{WORK_DIR}/Text2KGBench'
RESULT_DIR = f'{REPO_DIR}/resultados/piloto'
D19_DIR    = f'{REPO_DIR}/resultados/d19'
C2_DIR     = f'{REPO_DIR}/configs/c2'
C2_RAW_DIR = f'{REPO_DIR}/resultados/c2_inducao'
PV3_DIR    = f'{REPO_DIR}/resultados/piloto_v3'
VLLM_LOG   = f'{WORK_DIR}/vllm.log'
MODELO     = 'Qwen/Qwen2.5-7B-Instruct-AWQ'
PORT       = 8000

GIT_EMAIL  = 'felipemoraissoares@gmail.com'
GIT_NAME   = 'Felipe Morais (Kaggle)'

print(f'REPO_DIR:  {REPO_DIR}')
print(f'C2_DIR:    {C2_DIR}')
print(f'BENCH_COMMIT: {BENCH_COMMIT[:12]}...')

In [ ]:
# Celula 2 - Helpers
import subprocess

def run(cmd, check=True, **kw):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f'Falhou (rc={result.returncode}): {cmd[:120]}')
    return result

def git(cmd):
    return run(f'git -C "{REPO_DIR}" {cmd}', check=False)

In [ ]:
# Celula 3 - Clonar repositorios
if not os.path.exists(REPO_DIR):
    run(f'git clone --depth=1 {REPO_URL} {REPO_DIR}')
else:
    print('Repo ja existe - atualizando...')
    git('pull')

if not os.path.exists(BENCH_DIR):
    run(f'git init {BENCH_DIR}')
    run(f'git -C {BENCH_DIR} fetch --depth=1 {BENCH_URL} {BENCH_COMMIT}')
    run(f'git -C {BENCH_DIR} checkout FETCH_HEAD')
    actual = run(f'git -C {BENCH_DIR} rev-parse HEAD', check=False).stdout.strip()
    if actual != BENCH_COMMIT:
        print(f'AVISO: bench commit {actual[:12]} difere do esperado {BENCH_COMMIT[:12]}')
    else:
        print(f'Text2KGBench OK - commit {BENCH_COMMIT[:12]}')
else:
    print('Text2KGBench ja existe.')

git(f'config user.email "{GIT_EMAIL}"')
git(f'config user.name "{GIT_NAME}"')
print('Clones OK.')

In [ ]:
# Celula 4 - Restaurar resultados anteriores (sessao interrompida)
import glob, shutil

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(D19_DIR, exist_ok=True)
os.makedirs(C2_DIR, exist_ok=True)
os.makedirs(C2_RAW_DIR, exist_ok=True)
os.makedirs(PV3_DIR, exist_ok=True)

restaurados = 0
input_base = '/kaggle/input'
if os.path.exists(input_base):
    for dataset_name in os.listdir(input_base):
        for src in glob.glob(f'{input_base}/{dataset_name}/**/piloto_*.jsonl', recursive=True):
            dest = os.path.join(RESULT_DIR, os.path.basename(src))
            if not os.path.exists(dest):
                shutil.copy2(src, dest); restaurados += 1
                print(f'  Restaurado piloto: {os.path.basename(src)}')
        for src in glob.glob(f'{input_base}/{dataset_name}/**/d19_*.jsonl', recursive=True):
            dest = os.path.join(D19_DIR, os.path.basename(src))
            if not os.path.exists(dest):
                shutil.copy2(src, dest); restaurados += 1
                print(f'  Restaurado D19: {os.path.basename(src)}')
        for src in glob.glob(f'{input_base}/{dataset_name}/**/pv3_*.jsonl', recursive=True):
            dest = os.path.join(PV3_DIR, os.path.basename(src))
            if not os.path.exists(dest):
                shutil.copy2(src, dest); restaurados += 1
                print(f'  Restaurado PV3: {os.path.basename(src)}')
        for src in glob.glob(f'{input_base}/{dataset_name}/**/raw_*.jsonl', recursive=True):
            dest = os.path.join(C2_RAW_DIR, os.path.basename(src))
            if not os.path.exists(dest):
                shutil.copy2(src, dest); restaurados += 1
                print(f'  Restaurado C2 raw: {os.path.basename(src)}')
        for src in glob.glob(f'{input_base}/{dataset_name}/**/calibracao_*.json', recursive=True):
            dest = os.path.join(C2_DIR, os.path.basename(src))
            if not os.path.exists(dest):
                shutil.copy2(src, dest); restaurados += 1
                print(f'  Restaurado C2 calib: {os.path.basename(src)}')
        for src in glob.glob(f'{input_base}/{dataset_name}/**/definicoes_*.json', recursive=True):
            dest = os.path.join(C2_DIR, os.path.basename(src))
            if not os.path.exists(dest):
                shutil.copy2(src, dest); restaurados += 1
                print(f'  Restaurado C2 def: {os.path.basename(src)}')
        for src in glob.glob(f'{input_base}/{dataset_name}/**/*_induced.json', recursive=True):
            dest = os.path.join(C2_DIR, os.path.basename(src))
            if not os.path.exists(dest):
                shutil.copy2(src, dest); restaurados += 1
                print(f'  Restaurado C2 induced: {os.path.basename(src)}')

if restaurados > 0:
    print(f'\nTotal restaurado: {restaurados} arquivo(s).')
else:
    print('Nenhum resultado anterior encontrado - execucao completa.')

In [ ]:
# Celula 5 - Instalar dependencias
run('pip install -q vllm openai sentence-transformers scikit-learn nltk scipy numpy pandas statsmodels')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import sys
sys.path.insert(0, f'{REPO_DIR}/src')
print('Dependencias OK.')

In [ ]:
# Celula 6 - Subir vLLM em segundo plano
# dtype=half obrigatorio para T4 (fp16). tensor-parallel-size=2 usa ambas as GPUs T4.

vllm_cmd = (
    f'vllm serve {MODELO} '
    f'--port {PORT} '
    f'--dtype half '
    f'--tensor-parallel-size 2 '
    f'--max-model-len 4096 '
    f'--gpu-memory-utilization 0.90 '
    f'> {VLLM_LOG} 2>&1 &'
)
os.system(vllm_cmd)
print(f'vLLM iniciado em segundo plano. Log: {VLLM_LOG}')

In [ ]:
# Celula 7 - Aguardar vLLM pronto
import time, urllib.request, urllib.error

VLLM_URL  = f'http://localhost:{PORT}'
TIMEOUT   = 360
INTERVALO = 10
elapsed   = 0

while elapsed < TIMEOUT:
    try:
        urllib.request.urlopen(f'{VLLM_URL}/health', timeout=3)
        print(f'vLLM pronto apos {elapsed}s.')
        break
    except (urllib.error.URLError, ConnectionRefusedError, OSError):
        if elapsed % 30 == 0:
            print(f'  aguardando vLLM... ({elapsed}s / {TIMEOUT}s)')
        time.sleep(INTERVALO)
        elapsed += INTERVALO
else:
    print(f'ERRO: vLLM nao respondeu em {TIMEOUT}s. Ultimas linhas do log:')
    run(f'tail -40 {VLLM_LOG}', check=False)
    raise RuntimeError('vLLM nao iniciou. Verifique GPU e modelo.')

In [ ]:
# Celula 8 - Carregar manifesto de experimentos
# MANIFESTO aponta para o arquivo de configuracao de experimentos.
import json, os as _os

MANIFESTO = f'{REPO_DIR}/configs/pipeline/manifesto_piloto_v3.json'

RESULTS_DIR = PV3_DIR
_os.makedirs(RESULTS_DIR, exist_ok=True)

with open(MANIFESTO, encoding='utf-8') as f:
    exps = json.load(f)

# Ajusta caminhos de saida: relativo -> absoluto no Kaggle
for exp in exps:
    if 'saida' in exp:
        exp['saida'] = f"{RESULTS_DIR}/{_os.path.basename(exp['saida'])}"

manifesto_kaggle_path = f'{WORK_DIR}/manifesto_kaggle.json'
with open(manifesto_kaggle_path, 'w', encoding='utf-8') as f:
    json.dump(exps, f, ensure_ascii=False, indent=2)

print(f'Manifesto: {_os.path.basename(MANIFESTO)} ({len(exps)} experimentos)')
c2_no_manifesto = [e for e in exps if e.get('condicao') == 'C2']
print(f'Experimentos C2: {len(c2_no_manifesto)} ({[e["id"] for e in c2_no_manifesto]})')
for exp in exps:
    existe = '[ja existe]' if _os.path.exists(exp['saida']) else '[pendente]'
    print(f"  {exp['id']}  {existe}")

In [ ]:
# Celula 9 - C2: (a) extracao livre treino + (b) definicoes D-24 + (c) inducao EDC + (d) tabela + push
#
# Esta celula DEVE rodar antes dos experimentos (cel 11).
# Gera PV3_RUN_ID que o guard check (cel 10) usa para validar as ontologias.
import datetime

PV3_RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
print(f'PV3_RUN_ID = {PV3_RUN_ID}')

os.makedirs(C2_RAW_DIR, exist_ok=True)
manifesto_c2 = json.load(open(f'{REPO_DIR}/configs/pipeline/reinducao_c2_kaggle.json', encoding='utf-8'))
tarefas_ativas = [t for t in manifesto_c2 if t.get('ativo', True)]
print(f'Inducao C2: {len(tarefas_ativas)} tarefa(s) ativas de {len(manifesto_c2)} total')

PYPATH = {**os.environ, 'PYTHONPATH': f'{REPO_DIR}/src'}

for tarefa in tarefas_ativas:
    onto    = tarefa['onto']
    dataset = tarefa['dataset']
    seed    = tarefa['seed']
    raw_jsonl = f"{C2_RAW_DIR}/{os.path.basename(tarefa['saida_raw_jsonl'])}"

    # Invalida artefatos C2 anteriores para esta ontologia
    invalidos = [
        f'{C2_DIR}/{dataset}_{onto}_induced.json',
        f'{C2_DIR}/{dataset}_{onto}_frases_seed{seed}.json',
        f'{C2_DIR}/calibracao_{dataset}_{onto}.json',
    ]
    for p in invalidos:
        if os.path.exists(p):
            os.remove(p)
            print(f'  Removido (invalido): {os.path.basename(p)}')

    # (a) Extracao livre nas 50 frases de TREINO (gera raw JSONL + ontologia preliminar)
    print(f'\n=== (a) Extracao livre treino: {dataset}/{onto} ===')
    run(
        f'python {REPO_DIR}/scripts/induzir_c2.py '
        f"--dataset {tarefa['dataset']} "
        f"--onto {tarefa['onto']} "
        f'--url http://localhost:{PORT}/v1 '
        f'--modelo_llm {MODELO} '
        f"--threshold {tarefa['threshold']} "
        f"--freq_min {tarefa['freq_min']} "
        f"--n_frases {tarefa['n_frases']} "
        f"--seed {tarefa['seed']} "
        f"{'--calibrar' if tarefa.get('calibrar') else ''} "
        f'--saida_raw_jsonl {raw_jsonl} '
        f'--saida_dir {C2_DIR} '
        f'--normalizar '
        f'--run_id {PV3_RUN_ID}',
        cwd=REPO_DIR, env=PYPATH,
    )

    # (b) Geracao das definicoes (relacoes livres + relacoes da ontologia de referencia)
    print(f'\n=== (b) Definicoes D-24: {dataset}/{onto} ===')
    run(
        f'python {REPO_DIR}/scripts/gerar_definicoes_relacoes.py '
        f'--onto_id {onto} '
        f'--dataset {dataset} '
        f'--source_jsonl {raw_jsonl} '
        f'--url http://localhost:{PORT}/v1 '
        f'--modelo_llm {MODELO}',
        cwd=REPO_DIR, env=PYPATH,
    )

    # (c) Re-inducao com rotulo + definicao (EDC D-24), threshold=0.70, freq_min=3
    # Sobrescreve a ontologia gerada em (a) com a versao usando embeddings EDC
    print(f"\n=== (c) Re-inducao EDC (threshold={tarefa['threshold']}, freq_min={tarefa['freq_min']}): {dataset}/{onto} ===")
    run(
        f'python {REPO_DIR}/scripts/induzir_c2.py '
        f"--dataset {tarefa['dataset']} "
        f"--onto {tarefa['onto']} "
        f"--threshold {tarefa['threshold']} "
        f"--freq_min {tarefa['freq_min']} "
        f"--n_frases {tarefa['n_frases']} "
        f"--seed {tarefa['seed']} "
        f'--from_raw_jsonl {raw_jsonl} '
        f'--saida_dir {C2_DIR} '
        f'--normalizar '
        f'--use_definicoes '
        f'--run_id {PV3_RUN_ID}',
        cwd=REPO_DIR, env=PYPATH,
    )

    # (d) Tabela comparativa "so rotulo x rotulo + definicao"
    print(f'\n=== (d) Tabela calibracao: {dataset}/{onto} ===')
    run(
        f'python {REPO_DIR}/scripts/calibrar_c2_com_definicoes.py '
        f'--onto_id {onto} '
        f'--dataset {dataset} '
        f'--from_cache',
        cwd=REPO_DIR, env=PYPATH, check=False,
    )

print('\n=== Inducao C2 completa. ===')

# Push C2: definicoes + ontologias induzidas + raw JSONL
git('add resultados/c2_inducao/ configs/c2/')
r_status = git('status --porcelain resultados/c2_inducao/ configs/c2/')
if r_status.stdout.strip():
    git(f'commit -m "auto: C2 inducao D-21+D-24 piloto-v3 run_id={PV3_RUN_ID}"')
    r_push = git('push')
    if r_push.returncode == 0:
        print('Push C2 OK.')
    else:
        print('AVISO: push C2 falhou — verifique GITHUB_TOKEN.')
else:
    print('C2: nada novo para commitar (ja estava atualizado).')

In [ ]:
# Celula 10 - Guard check: verifica que todos os experimentos C2 tem ontologia desta sessao
#
# Para com RuntimeError se qualquer ontologia induzida estiver:
#   - ausente (arquivo nao encontrado)
#   - gerada em outra sessao (run_id != PV3_RUN_ID)
print(f'Guard check C2 (run_id esperado = {PV3_RUN_ID}) ...')
c2_exps = [e for e in exps if e.get('condicao') == 'C2']
print(f'  {len(c2_exps)} experimento(s) C2 no manifesto.')

erros = []
for exp in c2_exps:
    dataset   = exp['dataset']
    onto_id   = exp['ontologia_id']
    onto_path = f'{C2_DIR}/{dataset}_{onto_id}_induced.json'

    if not os.path.exists(onto_path):
        erros.append(f'AUSENTE: {onto_path}')
        continue

    with open(onto_path, encoding='utf-8') as _f:
        _data = json.load(_f)
    run_id_onto = _data.get('metadata', {}).get('run_id')

    if run_id_onto != PV3_RUN_ID:
        erros.append(
            f'RUN_ID INCORRETO: {os.path.basename(onto_path)}\n'
            f'  Encontrado : {run_id_onto!r}\n'
            f'  Esperado   : {PV3_RUN_ID!r}'
        )
    else:
        rels = [r['label'] for r in _data.get('relations', [])]
        print(f'  OK {dataset}/{onto_id}: {len(rels)} relacoes  ({", ".join(rels[:4])}...)')

if erros:
    raise RuntimeError(
        f'\n\nGuard check C2 falhou — {len(erros)} erro(s):\n\n'
        + '\n\n'.join(f'  * {e}' for e in erros)
        + '\n\nSolucao: execute novamente a celula de inducao C2 (celula 9) '
          'e confirme que PV3_RUN_ID coincide.'
    )

print(f'\nGuard check OK: {len(c2_exps)} experimento(s) C2 prontos para execucao.')

In [ ]:
# Celula 11 - Executar experimentos do manifesto
# FORCE_RERUN=True: remove os JSONLs de saida definidos no manifesto atual.
# Mude para False para retomar sessao interrompida.

FORCE_RERUN = True
if FORCE_RERUN:
    removidos = 0
    for exp in exps:
        saida = exp['saida']
        if os.path.exists(saida):
            os.remove(saida)
            removidos += 1
    if removidos:
        print(f'FORCE_RERUN=True: {removidos} JSONL(s) removidos.')
    else:
        print('FORCE_RERUN=True: nenhum JSONL encontrado (primeira execucao).')

PYPATH = {**os.environ, 'PYTHONPATH': f'{REPO_DIR}/src'}

run(
    f'python -m t2kg.run '
    f'--manifesto {manifesto_kaggle_path} '
    f'--bench_dir {BENCH_DIR} '
    f'--skip_existentes '
    f'--verbose',
    cwd=REPO_DIR,
    env=PYPATH
)
print('\nExperimentos concluidos.')

In [ ]:
# Celula 12 - Push final de todos os resultados
git('add resultados/piloto_v3/ resultados/c2_inducao/ configs/c2/')
r_status = git('status --porcelain resultados/piloto_v3/')
if r_status.stdout.strip():
    git('commit -m "auto: piloto-v3 completo (28 exp, workers=1, instrucao D, D-24)"')
    r_push = git('push')
    if r_push.returncode == 0:
        print('Push piloto-v3 OK.')
    else:
        raise RuntimeError('Push final falhou — verifique GITHUB_TOKEN e conectividade.')
else:
    print('Piloto-v3: nenhum resultado novo para commitar.')
    print('Se esperava resultados, verifique se os JSONLs foram gerados na celula 11.')

In [ ]:
# Celula 13 - Avaliar resultados piloto-v3 + push evidencias
AVALIACAO_PV3_DIR = f'{REPO_DIR}/resultados/avaliacao_piloto_v3'
os.makedirs(AVALIACAO_PV3_DIR, exist_ok=True)

PYPATH = {**os.environ, 'PYTHONPATH': f'{REPO_DIR}/src'}

# Avaliacao padrao + metricas estendidas (C0 projetada, C2/C3 projecao, redundancia)
run(
    f'python {REPO_DIR}/scripts/avaliar_resultados.py '
    f'--manifesto {manifesto_kaggle_path} '
    f'--bench_dir {BENCH_DIR} '
    f'--avaliacao_dir {AVALIACAO_PV3_DIR} '
    f'--csv {AVALIACAO_PV3_DIR}/tabela_piloto_v3.csv '
    f'--extended',
    cwd=REPO_DIR, env=PYPATH,
)

# Push avaliacao
git(f'add resultados/avaliacao_piloto_v3/')
r_status = git('status --porcelain resultados/avaliacao_piloto_v3/')
if r_status.stdout.strip():
    git('commit -m "auto: avaliacao piloto-v3 (28 exp, Qwen2.5-7B, D-24)"')
    r_push = git('push')
    if r_push.returncode == 0:
        print('Push avaliacao OK.')
    else:
        raise RuntimeError('Push avaliacao falhou — verifique GITHUB_TOKEN.')
else:
    print('Avaliacao: nenhum resultado novo para commitar.')

print(f'\nEvidencias em: resultados/avaliacao_piloto_v3/')

In [ ]:
# Celula 13 - Verificar log do vLLM (diagnostico)
run(f'tail -30 {VLLM_LOG}', check=False)